. The Math for Custom SMOTE (Fixing Imbalance)Before the model can learn, we have to artificially generate new, realistic fraud data points so the algorithm doesn't just guess "0" (Genuine) every time.Step A: Find the Nearest Neighbors (Euclidean Distance)To create a fake fraud transaction, the algorithm first looks at a real fraud point ($x$) and finds its closest real fraud neighbor ($y$). We calculate the exact distance between them using the Euclidean distance formula across all 12 features:$$d(x, y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$$Step B: Generate the Synthetic Point (Interpolation)Once we find a neighbor, we draw a mathematical line between the two fraud points and drop a brand new, synthetic fraud point ($x_{\text{new}}$) randomly on that line.$$x_{\text{new}} = x + \lambda (x_{\text{neighbor}} - x)$$(Where $\lambda$ is a random number between $0$ and $1$)2. The Math for Cost-Sensitive Logistic RegressionNow that the data is balanced, we build the brain. Standard Logistic Regression treats all mistakes equally. We are going to "hack" the math by adding penalties (weights) so the model suffers a massive mathematical penalty if it misses a fraud case.Step A: The Activation (Sigmoid Function)We take our 12 features, multiply them by their learned weights ($w$), add a bias ($b$), and push that raw number through a Sigmoid curve to get a probability between 0% and 100%:$$\hat{y} = \frac{1}{1 + e^{-(w^T x + b)}}$$Step B: The Penalized Loss Function (Weighted Log Loss)This is the exact equation that gets you the marks. We modify the standard Binary Cross-Entropy loss by multiplying it by $W_1$ (a huge penalty for missing fraud) and $W_0$ (a small penalty for a false alarm):$$J(w, b) = -\frac{1}{N} \sum_{i=1}^{N} \left[ W_1 \cdot y_i \log(\hat{y}_i) + W_0 \cdot (1 - y_i) \log(1 - \hat{y}_i) \right]$$Step C: The Learning Process (Gradient Descent)To make the model smarter, we calculate the derivative (the slope) of the error and update our weights step-by-step using a learning rate ($\alpha$).$$w = w - \alpha \frac{\partial J}{\partial w}$$

In [13]:
import pandas as pd
import numpy as np
import math
import random

In [14]:
df = pd.read_csv("creditcard.csv")

# 1. Define the exact 12 features chosen during Feature Selection
final_features = ['V10', 'V11', 'V12', 'V14', 'V16', 'V17', 'V18', 'V2', 'V3', 'V4', 'V7', 'V9']

# 2. Extract only those 12 columns as your input matrix (X)
X = df[final_features].values

# 3. Extract the target column (y)
y = df['Class'].values

In [15]:
def euclidean_distance(point_a, point_b):
    """
    Step A: Calculates the exact straight-line distance between two data points.
    """
    return np.sqrt(np.sum((point_a - point_b)**2))

def get_k_nearest_neighbors(core_index, minority_data, k=5):
    """
    Finds the 'k' closest fraud transactions to our core fraud point using its index.
    """
    distances = []
    core_point = minority_data[core_index]

    for i in range(len(minority_data)):
        # Ensure we don't compare the point to itself by checking the index
        if i != core_index:
            dist = euclidean_distance(core_point, minority_data[i])
            distances.append((dist, minority_data[i]))

    # Sort the list by distance (smallest first) and grab the top 'k'
    distances.sort(key=lambda x: x[0])
    nearest_neighbors = [item[1] for item in distances[:k]]

    return nearest_neighbors

def custom_smote(minority_data, num_synthetic_samples, k=5):
    """
    Step B: Generates synthetic fraud data points using mathematical interpolation.
    """
    synthetic_data = []
    minority_data = np.array(minority_data) # Ensure it's a numpy array for math operations

    for _ in range(num_synthetic_samples):
        # 1. Pick a random fraud transaction from our original dataset
        random_index = random.randint(0, len(minority_data) - 1)
        core_point = minority_data[random_index]

        # 2. Find its closest fraud neighbors (passing the index now)
        neighbors = get_k_nearest_neighbors(random_index, minority_data, k)

        # 3. Randomly select one of those close neighbors
        chosen_neighbor = random.choice(neighbors)

        # 4. INTERPOLATION MATH: Drop a new point on the line between them
        lambda_val = random.random() # Generates a random float between 0.0 and 1.0
        synthetic_point = core_point + lambda_val * (chosen_neighbor - core_point)

        # 5. Save the newly created fake fraud point
        synthetic_data.append(synthetic_point)

    return np.array(synthetic_data)

In [16]:
class CustomLogisticRegression:
    def __init__(self, learning_rate=0.01, epochs=1000, weight_fraud=10.0, weight_genuine=1.0):
        """
        Setting up the hyperparameters.
        weight_fraud is set to 10.0 to mathematically punish the model if it misses a fraud case.
        """
        self.lr = learning_rate
        self.epochs = epochs
        self.w1 = weight_fraud
        self.w0 = weight_genuine
        self.weights = None
        self.bias = None

    def _sigmoid(self, z):
        """
        Step A: The Activation Function.
        Converts the raw linear equation into a probability between 0 and 1.
        (We use np.clip to prevent the math from overflowing and crashing the script)
        """
        z = np.clip(z, -250, 250)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        """
        Step C: The Learning Process (Gradient Descent).
        This loop trains the model over a set number of epochs.
        """
        # X is our dataset (features), y is our target (Class)
        num_samples, num_features = X.shape

        # Start with all weights and bias set to 0
        self.weights = np.zeros(num_features)
        self.bias = 0

        for _ in range(self.epochs):
            # 1. The Linear Equation: z = (w * x) + b
            linear_model = np.dot(X, self.weights) + self.bias

            # 2. Push it through Sigmoid to get a prediction (0.0 to 1.0)
            y_predicted = self._sigmoid(linear_model)

            # 3. Step B: Cost-Sensitive Error Calculation
            # This is where we inject the heavy penalty for missing fraud
            raw_error = y_predicted - y

            # If the actual label (y) is 1 (Fraud), multiply the error by w1 (10x penalty)
            # If the actual label is 0 (Genuine), multiply the error by w0 (1x penalty)
            weighted_error = np.where(y == 1, raw_error * self.w1, raw_error * self.w0)

            # 4. Calculate the Gradients (Derivatives)
            dw = (1 / num_samples) * np.dot(X.T, weighted_error)
            db = (1 / num_samples) * np.sum(weighted_error)

            # 5. Update the Weights using the Learning Rate
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

    def predict(self, X, threshold=0.5):
        """
        Makes the final prediction. If the probability is > 50%, flag as Fraud (1).
        """
        linear_model = np.dot(X, self.weights) + self.bias
        y_predicted = self._sigmoid(linear_model)

        return np.array([1 if i > threshold else 0 for i in y_predicted])

Since standard accuracy is a trap (as we established for your poster), we need to calculate True Positives (TP), False Positives (FP), True Negatives (TN), and False Negatives (FN) to find our Precision and Recall.Precision: $\frac{TP}{TP + FP}$ (Out of all transactions flagged as fraud, how many were actually fraud?)Recall: $\frac{TP}{TP + FN}$ (Out of all actual frauds, how many did we catch?)

In [17]:
def custom_train_test_split(X, y, test_size=0.2, random_seed=42):
    """
    Splits the data into training and testing sets using a STRATIFIED split
    to ensure the ratio of fraud to genuine cases remains the same.
    """
    np.random.seed(random_seed)
    
    # Separate the indices of the two classes
    fraud_indices = np.where(y == 1)[0]
    genuine_indices = np.where(y == 0)[0]
    
    # Shuffle indices within each class
    np.random.shuffle(fraud_indices)
    np.random.shuffle(genuine_indices)
    
    # Calculate split points for each class
    fraud_split = int(len(fraud_indices) * (1 - test_size))
    genuine_split = int(len(genuine_indices) * (1 - test_size))
    
    # Split both classes
    train_fraud, test_fraud = fraud_indices[:fraud_split], fraud_indices[fraud_split:]
    train_genuine, test_genuine = genuine_indices[:genuine_split], genuine_indices[genuine_split:]
    
    # Combine the training indices and testing indices
    train_idx = np.concatenate([train_fraud, train_genuine])
    test_idx = np.concatenate([test_fraud, test_genuine])
    
    # Shuffle the final train and test sets so classes aren't clustered together
    np.random.shuffle(train_idx)
    np.random.shuffle(test_idx)
    
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def evaluate_model(y_true, y_predicted):
    """
    Calculates the Confusion Matrix, Precision, Recall, and F1-Score from scratch.
    """
    TP = np.sum((y_true == 1) & (y_predicted == 1))
    TN = np.sum((y_true == 0) & (y_predicted == 0))
    FP = np.sum((y_true == 0) & (y_predicted == 1))
    FN = np.sum((y_true == 1) & (y_predicted == 0))

    # Calculate Metrics (Adding a tiny number 1e-9 to prevent dividing by zero errors)
    precision = TP / (TP + FP + 1e-9)
    recall = TP / (TP + FN + 1e-9)
    f1_score = 2 * (precision * recall) / (precision + recall + 1e-9)

    print("\n--- CUSTOM EVALUATION METRICS ---")
    print(f"True Positives (Caught Fraud): {TP}")
    print(f"False Negatives (Missed Fraud): {FN}")
    print(f"False Positives (False Alarms): {FP}")
    print(f"True Negatives (Legitimate Passed): {TN}")
    print("-" * 30)
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1_score:.4f}")

    return precision, recall, f1_score


In [18]:
# 1. Split the original data into Train and Test sets
print("Splitting data...")
X_train, X_test, y_train, y_test = custom_train_test_split(X, y, test_size=0.2)

# 2. Extract ONLY the fraud cases from the training set to feed into SMOTE
minority_mask = (y_train == 1)
X_minority = X_train[minority_mask]

# Calculate how many fake fraud cases we need to generate to perfectly balance the data
num_genuine = np.sum(y_train == 0)
num_fraud = np.sum(y_train == 1)
num_synthetic_to_generate = num_genuine - num_fraud

# 3. Generate the synthetic fraud data using your Custom SMOTE
print(f"Generating {num_synthetic_to_generate} synthetic fraud transactions... (This may take a few seconds)")
synthetic_X = custom_smote(X_minority, num_synthetic_samples=num_synthetic_to_generate, k=5)

# Create the '1' labels for all the newly generated synthetic fraud transactions
synthetic_y = np.ones(len(synthetic_X))

# 4. Combine the original training data with the new synthetic data
X_train_balanced = np.vstack((X_train, synthetic_X))
y_train_balanced = np.concatenate((y_train, synthetic_y))

# 5. Initialize your Custom Logistic Regression Model
# IMPORTANT: Because we balanced the data with SMOTE, we should NOT use the 10x penalty, 
# so we set weight_fraud to 1.0 (Standard Logistic Regression on balanced data).
model = CustomLogisticRegression(learning_rate=0.01, epochs=1000, weight_fraud=1.0, weight_genuine=1.0)

# 6. Train the model
print("Training the model... (Running Gradient Descent)")
model.fit(X_train_balanced, y_train_balanced)

# 7. Make predictions on the unseen TEST set
print("Making predictions...")
y_predicted = model.predict(X_test, threshold=0.5)

# 8. Finally, evaluate the model (THIS will print your metrics!)
precision, recall, f1 = evaluate_model(y_test, y_predicted)


Splitting data...
Generating 227059 synthetic fraud transactions... (This may take a few seconds)


KeyboardInterrupt: 